In [1]:
import Pkg;
Pkg.activate(@__DIR__);
Pkg.status()
Pkg.instantiate()

  Activating project at `~/Desktop/Evolutionary-Algos/rdex/src`


Status `~/Desktop/Evolutionary-Algos/rdex/src/Project.toml`
  [09f84164] HypothesisTests v0.11.6


In [2]:
using DelimitedFiles
using Statistics
using Printf

In [3]:
algorithms = Dict(
    "Random" => "NoMods",
    "Sobol"  => "SobolInit",
    "Halton" => "HaltonInit",
    "ExtArch" => "ExtArchive"
    "ExtArch2" => "ExtArchive2"
)

num_funcs = 28
runs = 25

struct Trial
    alg::String
    ev::Float64
    cv::Float64
    fe::Float64
end

In [4]:
function load_trials(folder, alg, f)

    ffile = joinpath(folder, @sprintf("RDEx_f_F%d_D30_.txt", f))
    cfile = joinpath(folder, @sprintf("RDEx_C_F%d_D30.txt", f))
    sfile = joinpath(folder, @sprintf("RDEx_F%d_D30.txt", f))

    fdata = readdlm(ffile)
    cdata = readdlm(cfile)
    sdata = readdlm(sfile)

    trials = Trial[]

    for r in 1:runs

        ev = fdata[r,end]

        cv = sum(abs.(cdata[r,end-8:end]))

        fe = sdata[r,end]

        push!(trials, Trial(alg,ev,cv,fe))

    end

    return trials

end

load_trials (generic function with 1 method)

In [5]:
function accuracy_score(t1,t2)

    feasible1 = t1.cv == 0
    feasible2 = t2.cv == 0

    if feasible1 && feasible2

        if t1.ev < t2.ev
            return (1,0)
        elseif t1.ev > t2.ev
            return (0,1)
        else
            return (0.5,0.5)
        end

    elseif !feasible1 && !feasible2

        if t1.cv < t2.cv
            return (1,0)
        elseif t1.cv > t2.cv
            return (0,1)
        else
            return (0.5,0.5)
        end

    elseif feasible1
        return (1,0)
    else
        return (0,1)
    end

end

accuracy_score (generic function with 1 method)

In [6]:
function speed_score(t1,t2)

    if t1.fe < t2.fe
        return (1,0)
    elseif t1.fe > t2.fe
        return (0,1)
    else
        return (0.5,0.5)
    end

end

speed_score (generic function with 1 method)

In [7]:
println("Scores")
println("------------------------------------------------")

total_scores = Dict(a=>0.0 for a in keys(algorithms))

for f in 1:num_funcs

    trials = Trial[]

    for (name,path) in algorithms
        append!(trials, load_trials(path,name,f))
    end

    scores = Dict(a=>0.0 for a in keys(algorithms))

    n = length(trials)

    for i in 1:n-1
        for j in i+1:n

            t1 = trials[i]
            t2 = trials[j]

            a1,a2 = accuracy_score(t1,t2)
            s1,s2 = speed_score(t1,t2)

            scores[t1.alg] += a1 + s1
            scores[t2.alg] += a2 + s2

        end
    end

    println("F$f")

    for alg in keys(scores)
        @printf("  %-7s : %.1f\n", alg, scores[alg])
        total_scores[alg] += scores[alg]
    end

end

Scores
------------------------------------------------
F1
  Halton  : 3567.0
  Random  : 3883.0
  Sobol   : 1111.0
  ExtArch : 1339.0
F2
  Halton  : 3500.5
  Random  : 3946.5
  Sobol   : 1583.0
  ExtArch : 870.0
F3
  Halton  : 2475.0
  Random  : 2475.0
  Sobol   : 2475.0
  ExtArch : 2475.0
F4
  Halton  : 2475.0
  Random  : 2475.0
  Sobol   : 2475.0
  ExtArch : 2475.0
F5
  Halton  : 3574.0
  Random  : 3717.0
  Sobol   : 827.0
  ExtArch : 1782.0
F6
  Halton  : 3304.5
  Random  : 3570.0
  Sobol   : 1671.0
  ExtArch : 1354.5
F7
  Halton  : 3236.0
  Random  : 2822.0
  Sobol   : 1194.5
  ExtArch : 2647.5
F8
  Halton  : 3944.0
  Random  : 3506.0
  Sobol   : 1662.0
  ExtArch : 788.0
F9
  Halton  : 1639.0
  Random  : 2317.0
  Sobol   : 1594.0
  ExtArch : 4350.0
F10
  Halton  : 3567.0
  Random  : 3868.0
  Sobol   : 973.0
  ExtArch : 1492.0
F11
  Halton  : 1438.5
  Random  : 1943.0
  Sobol   : 2192.5
  ExtArch : 4326.0
F12
  Halton  : 2475.0
  Random  : 2475.0
  Sobol   : 2475.0
  ExtArch : 2475

In [8]:
println("\nRanking")
println("--------------------------------")

sorted = sort(collect(total_scores), by=x->-x[2])

for (alg,score) in sorted
    @printf("%-8s : %.1f\n", alg, score)
end


Ranking
--------------------------------
Random   : 78206.5
Halton   : 75573.5
ExtArch  : 66877.0
Sobol    : 56543.0
